In [ ]:
%load_ext autoreload

%autoreload 2

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

# Inspect Dataset — Snapshot Safari SER (Serengeti)

Visual inspection of the curated SER camera trap subset.  
Covers: class distribution, MegaDetector confidence, resolution spread, sample grids (full-frame + cropped), full vs. cropped side-by-side, IR/night detection, ImageFolder loading, DataLoader batch.

## Imports

In [ ]:
import csv
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pyrootutils
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

## Parameters

In [ ]:
ROOT = pyrootutils.setup_root(
    search_from=".",
    indicator="pyproject.toml",
    project_root_env_var=True,
    dotenv=True,
    pythonpath=True,
    cwd=True,
)

SER_ROOT = ROOT / "data/ser"
BALANCED_DIR = SER_ROOT / "ser_balanced"
BALANCED_CROPPED_DIR = SER_ROOT / "ser_balanced_cropped"
SAMPLED_DIR = SER_ROOT / "ser_sampled"
SAMPLED_CROPPED_DIR = SER_ROOT / "ser_sampled_cropped"
SPLITS = ["train", "val", "test"]
SEED = 0

assert BALANCED_DIR.exists(), f"Not found: {BALANCED_DIR.resolve()}"
assert BALANCED_CROPPED_DIR.exists(), f"Not found: {BALANCED_CROPPED_DIR.resolve()}"
print(f"ser_balanced:         {BALANCED_DIR.resolve()}")
print(f"ser_balanced_cropped: {BALANCED_CROPPED_DIR.resolve()}")
print(f"ser_sampled:          {SAMPLED_DIR.resolve()}  (exists={SAMPLED_DIR.exists()})")
print(
    f"ser_sampled_cropped:  {SAMPLED_CROPPED_DIR.resolve()}  (exists={SAMPLED_CROPPED_DIR.exists()})"
)

## Section 1 — Class Distribution

Count images per class and split from `metadata.csv`, then plot a bar chart.

In [ ]:
with (BALANCED_DIR / "metadata.csv").open() as f:
    rows = list(csv.DictReader(f))

print(f"Total images: {len(rows)}")
print(f"Columns: {list(rows[0].keys())}\n")

counts = defaultdict(Counter)  # counts[split][label]
for r in rows:
    counts[r["split"]][r["label"]] += 1

all_labels = sorted({r["label"] for r in rows})
col_w = 8
print(
    f"{'label':<20}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'total':>{col_w}}"
)
print("-" * (20 + col_w * (len(SPLITS) + 1)))
for label in all_labels:
    cs = [counts[s][label] for s in SPLITS]
    print(f"{label:<20}" + "".join(f"{c:>{col_w}}" for c in cs) + f"{sum(cs):>{col_w}}")
totals = [sum(counts[s].values()) for s in SPLITS]
print(
    f"{'TOTAL':<20}"
    + "".join(f"{t:>{col_w}}" for t in totals)
    + f"{sum(totals):>{col_w}}"
)

In [ ]:
train_counts = [counts["train"][label] for label in all_labels]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(all_labels, train_counts, color="steelblue", edgecolor="white")
_ = ax.bar_label(bars, fmt="%d", padding=3)
_ = ax.set_title("SER (Serengeti) — train images per class", fontsize=13)
_ = ax.set_ylabel("Image count")
_ = ax.set_xlabel("Class")
_ = plt.xticks(rotation=20, ha="right")
_ = plt.tight_layout()
plt.show()

max_c, min_c = max(train_counts), min(train_counts)
print(f"Max/min ratio (train): {max_c}/{min_c} = {max_c / min_c:.1f}x")

## Section 2 — MegaDetector Confidence Distribution

All animal-class images were selected using MegaDetector (conf ≥ 0.8). The empty class
intentionally has `md_conf = 0` (no animal detected). Inspect the confidence distribution
for animal images to understand filtering quality.

In [ ]:
# Exclude empty class (md_conf == 0 by construction)
animal_confs = [
    float(r["md_conf"])
    for r in rows
    if r["label"] != "empty" and float(r["md_conf"]) > 0
]

print(f"Animal images with md_conf > 0: {len(animal_confs)}")
print(f"  min:    {min(animal_confs):.3f}")
print(f"  max:    {max(animal_confs):.3f}")
print(f"  mean:   {sum(animal_confs) / len(animal_confs):.3f}")
print(f"  median: {sorted(animal_confs)[len(animal_confs) // 2]:.3f}")

fig, ax = plt.subplots(figsize=(9, 3))
_ = sns.histplot(animal_confs, bins=40, color="steelblue", ax=ax)
_ = ax.axvline(0.8, color="red", linestyle="--", label="threshold (0.8)")
_ = ax.set_title("MegaDetector animal confidence — SER animal classes", fontsize=12)
_ = ax.set_xlabel("Confidence")
_ = ax.set_ylabel("Images")
_ = ax.legend()
_ = plt.tight_layout()
plt.show()

## Section 3 — Resolution Distribution

All images were resized to max 1024 px on the longer side (aspect-ratio preserving).
Confirm the spread and check for any unexpectedly small images.

In [ ]:
widths = [int(r["width"]) for r in rows]
heights = [int(r["height"]) for r in rows]

print(f"Width  — min: {min(widths)}, max: {max(widths)}, unique: {len(set(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, unique: {len(set(heights))}")

longer_sides = [max(w, h) for w, h in zip(widths, heights, strict=False)]
print(f"\nLonger side — min: {min(longer_sides)}, max: {max(longer_sides)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
_ = sns.histplot(widths, bins=20, color="steelblue", ax=axes[0])
_ = axes[0].set_title("Width distribution")
_ = axes[0].set_xlabel("Pixels")
_ = sns.histplot(heights, bins=20, color="coral", ax=axes[1])
_ = axes[1].set_title("Height distribution")
_ = axes[1].set_xlabel("Pixels")
_ = plt.suptitle("SER — image resolution (max 1024 px on longer side)", fontsize=12)
_ = plt.tight_layout()
plt.show()

## Section 4 — Sample Grid per Class (Full Frames)

4 × 4 random training samples per class from the full camera-trap frames.  
Check for: wrong labels, IR/night images (expected ~56%), extreme exposure, empty frames.

In [ ]:
random.seed(SEED)

train_index = defaultdict(list)
for label_dir in sorted((BALANCED_DIR / "train").iterdir()):
    if label_dir.is_dir():
        train_index[label_dir.name] = sorted(label_dir.iterdir())

N_COLS, N_ROWS = 4, 4
for label in all_labels:
    paths = train_index.get(label, [])
    sample = random.sample(paths, min(N_COLS * N_ROWS, len(paths)))
    n = len(sample)
    n_rows = (n + N_COLS - 1) // N_COLS

    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3, n_rows * 2.5))
    fig.suptitle(f"class: {label}  (n_train={len(paths)})", fontsize=12, y=1.01)
    axes_flat = axes.flatten() if n > 1 else [axes]

    for ax, p in zip(axes_flat, sample, strict=False):
        with Image.open(p) as im:
            _ = ax.imshow(im)
        _ = ax.axis("off")
    for ax in axes_flat[n:]:
        _ = ax.axis("off")

    _ = plt.tight_layout()
    plt.show()

## Section 5 — Sample Grid per Class (Cropped)

4 × 4 random training samples per animal class from `ser_balanced_cropped/` — images cropped
to the primary MegaDetector bounding box with 10% padding.
Empty class has no crops (no bounding box available).


In [ ]:
random.seed(SEED)

cropped_index = defaultdict(list)
for label_dir in sorted((BALANCED_CROPPED_DIR / "train").iterdir()):
    if label_dir.is_dir():
        cropped_index[label_dir.name] = sorted(label_dir.iterdir())

animal_labels = [lbl for lbl in all_labels if lbl != "empty"]

for label in animal_labels:
    paths = cropped_index.get(label, [])
    sample = random.sample(paths, min(N_COLS * N_ROWS, len(paths)))
    n = len(sample)
    n_rows = (n + N_COLS - 1) // N_COLS

    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3, n_rows * 2.5))
    fig.suptitle(
        f"[cropped] class: {label}  (n_train={len(paths)})", fontsize=12, y=1.01
    )
    axes_flat = axes.flatten() if n > 1 else [axes]

    for ax, p in zip(axes_flat, sample, strict=False):
        with Image.open(p) as im:
            _ = ax.imshow(im)
        _ = ax.axis("off")
    for ax in axes_flat[n:]:
        _ = ax.axis("off")

    _ = plt.tight_layout()
    plt.show()

## Section 6 — Full vs. Cropped Side-by-Side

For each of the first four animal classes, display 4 matched pairs:  
top row = full camera-trap frame, bottom row = MegaDetector crop.  
Illustrates how MegaDetector box quality varies across species and lighting conditions.

In [ ]:
random.seed(SEED)

N_PAIRS = 4

for label in animal_labels[:4]:
    full_paths = []
    for split in SPLITS:
        full_paths.extend((BALANCED_DIR / split / label).glob("*.jpg"))

    sample_full = random.sample(full_paths, min(N_PAIRS, len(full_paths)))

    fig, axes = plt.subplots(2, N_PAIRS, figsize=(N_PAIRS * 3, 6))
    fig.suptitle(f"{label} — full frame (top) vs. MD crop (bottom)", fontsize=11)

    for col, p_full in enumerate(sample_full):
        stem = p_full.stem
        p_crop = None
        for split in SPLITS:
            candidate = BALANCED_CROPPED_DIR / split / label / f"{stem}.jpg"
            if candidate.exists():
                p_crop = candidate
                break

        _ = axes[0, col].axis("off")
        _ = axes[1, col].axis("off")

        with Image.open(p_full) as im:
            _ = axes[0, col].imshow(np.array(im))
            _ = axes[0, col].set_title(f"{im.width}x{im.height}", fontsize=8)

        if p_crop and p_crop.exists():
            with Image.open(p_crop) as im:
                _ = axes[1, col].imshow(np.array(im))
                _ = axes[1, col].set_title(f"{im.width}x{im.height}", fontsize=8)
        else:
            axes[1, col].text(
                0.5,
                0.5,
                "no crop",
                ha="center",
                va="center",
                transform=axes[1, col].transAxes,
            )

    _ = plt.tight_layout()
    plt.show()

## Section 7 — IR / Night Image Detection

Camera traps capture many night-time images using infrared illumination (nearly greyscale).
We flag images where the mean absolute deviation of per-channel RGB means falls below a
threshold — a simple proxy for low colour saturation.

In [ ]:
IR_THRESHOLD = 5.0  # mean abs deviation of RGB channel means
IR_SAMPLE = 200

all_jpgs = list(BALANCED_DIR.rglob("*.jpg"))
random.seed(SEED)
sample_jpgs = random.sample(all_jpgs, min(IR_SAMPLE, len(all_jpgs)))

ir_count = 0
for p in sample_jpgs:
    with Image.open(p) as im:
        arr = np.array(im.convert("RGB"), dtype=float)
        channel_means = arr.mean(axis=(0, 1))
        saturation = float(np.abs(channel_means - channel_means.mean()).mean())
        if saturation < IR_THRESHOLD:
            ir_count += 1

ir_frac = ir_count / len(sample_jpgs)
print(f"IR/night images (saturation < {IR_THRESHOLD}):")
print(f"  {ir_count} / {len(sample_jpgs)} sampled  ({ir_frac:.0%})")
if ir_frac > 0.3:
    print("WARNING: >30% of images appear to be IR/night.")
else:
    print("OK: IR fraction is manageable.")

## Section 8 — ImageFolder Loading

Verify that torchvision's `ImageFolder` reads the full-frame dataset correctly and that
class indices are consistent across splits.

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_ds = ImageFolder(BALANCED_DIR / "train", transform=transform)
val_ds = ImageFolder(BALANCED_DIR / "val", transform=transform)
test_ds = ImageFolder(BALANCED_DIR / "test", transform=transform)

print(f"Classes ({len(train_ds.classes)}): {train_ds.classes}")
print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")
print(f"Test size:  {len(test_ds)}")

img, label_idx = train_ds[0]
print(f"\nSample tensor — shape: {img.shape}, dtype: {img.dtype}")
print(f"Value range:  [{img.min():.2f}, {img.max():.2f}]  (normalised, not [0,1])")

## Section 9 — DataLoader Batch

Load one batch and display it as a grid (after un-normalising for visualisation).

In [ ]:
loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

imgs, labels = next(iter(loader))
print(f"Batch shape:  {imgs.shape}")
print(f"Labels:       {[train_ds.classes[label_idx] for label_idx in labels.tolist()]}")

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
imgs_disp = (imgs * std + mean).clamp(0, 1)

grid = make_grid(imgs_disp, nrow=8, padding=4)
fig, ax = plt.subplots(figsize=(18, 5))
_ = ax.imshow(grid.permute(1, 2, 0).numpy())
_ = ax.set_title(
    "SER — one training batch (224x224, displayed after un-normalising)",
    fontsize=12,
)
_ = ax.axis("off")
_ = plt.tight_layout()
plt.show()

## Section 10 — Sanity Checks

Verify that file counts on disk match the manifest, and that no images are corrupt in
either the full-frame or cropped datasets.

In [ ]:
# File count vs manifest (scoped to ser_balanced only)
disk_balanced = list(BALANCED_DIR.rglob("*.jpg"))
disk_balanced_cropped = list(BALANCED_CROPPED_DIR.rglob("*.jpg"))
manifest_animal = [r for r in rows if r["label"] != "empty"]

print(
    f"ser_balanced/         — files: {len(disk_balanced):>5},  manifest entries: {len(rows):>5}"
)
print(
    f"ser_balanced_cropped/ — files: {len(disk_balanced_cropped):>5},  manifest (animal): {len(manifest_animal):>5}"
)

assert len(disk_balanced) == len(rows), "Mismatch: ser_balanced/ disk vs manifest"
assert len(disk_balanced_cropped) == len(manifest_animal), (
    "Mismatch: ser_balanced_cropped/ disk vs manifest"
)
print("OK — counts match")

In [ ]:
corrupt = []
for p in disk_balanced + disk_balanced_cropped:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        corrupt.append((p, str(e)))

print(f"Corrupt files: {len(corrupt)}")
for p, err in corrupt[:5]:
    print(f"  {p.name}: {err}")
assert len(corrupt) == 0, f"{len(corrupt)} corrupt files found"
print("OK — all images are readable")

## Section 11 — Distribution Comparison: ser_balanced vs. ser_sampled

Compare class distributions between the balanced (≤200/class) and sampled (≤1000/class,
5 000 total) variants. The sampled set better reflects the actual training data size
while still covering all 10 classes.


In [ ]:
if not SAMPLED_DIR.exists():
    print(
        "ser_sampled not yet downloaded — skipping. Run 10_download_sampled.py first."
    )
else:
    with (SAMPLED_DIR / "metadata.csv").open() as f:
        rows_sampled = list(csv.DictReader(f))

    counts_sampled = defaultdict(Counter)
    for r in rows_sampled:
        counts_sampled[r["split"]][r["label"]] += 1

    sampled_labels = sorted({r["label"] for r in rows_sampled})

    # Side-by-side bar chart
    x = range(len(all_labels))
    balanced_train = [counts["train"][lbl] for lbl in all_labels]
    sampled_train = [counts_sampled["train"][lbl] for lbl in all_labels]

    fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=False)

    bars0 = axes[0].bar(
        all_labels, balanced_train, color="steelblue", edgecolor="white"
    )
    axes[0].bar_label(bars0, fmt="%d", padding=2, fontsize=8)
    _ = axes[0].set_title("ser_balanced — train counts", fontsize=12)
    axes[0].set_ylabel("Images")
    axes[0].tick_params(axis="x", rotation=30)

    bars1 = axes[1].bar(all_labels, sampled_train, color="coral", edgecolor="white")
    axes[1].bar_label(bars1, fmt="%d", padding=2, fontsize=8)
    _ = axes[1].set_title("ser_sampled — train counts", fontsize=12)
    axes[1].set_ylabel("Images")
    axes[1].tick_params(axis="x", rotation=30)

    _ = plt.suptitle("Class distribution: balanced vs. sampled", fontsize=13)
    _ = plt.tight_layout()
    plt.show()

    # Summary table
    print(f"\n{'label':<20} {'balanced':>10} {'sampled':>10} {'ratio':>8}")
    print("-" * 50)
    for lbl in all_labels:
        b = sum(counts[s][lbl] for s in SPLITS)
        s = sum(counts_sampled[sp][lbl] for sp in SPLITS)
        ratio = f"{s / b:.1f}x" if b > 0 else "-"
        print(f"{lbl:<20} {b:>10} {s:>10} {ratio:>8}")
    b_total = sum(sum(counts[sp].values()) for sp in SPLITS)
    s_total = len(rows_sampled)
    print(f"{'TOTAL':<20} {b_total:>10} {s_total:>10}")

## Section 12 — ImageFolder Loading: ser_sampled

Verify that `ImageFolder` reads the sampled variant correctly.


In [ ]:
if not SAMPLED_DIR.exists():
    print("ser_sampled not yet downloaded — skipping.")
else:
    train_ds_s = ImageFolder(SAMPLED_DIR / "train", transform=transform)
    val_ds_s = ImageFolder(SAMPLED_DIR / "val", transform=transform)
    test_ds_s = ImageFolder(SAMPLED_DIR / "test", transform=transform)

    print(f"Classes ({len(train_ds_s.classes)}): {train_ds_s.classes}")
    print(f"Train size: {len(train_ds_s)}")
    print(f"Val size:   {len(val_ds_s)}")
    print(f"Test size:  {len(test_ds_s)}")

    loader_s = DataLoader(
        train_ds_s,
        batch_size=16,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )
    imgs_s, labels_s = next(iter(loader_s))
    print(f"\nBatch shape: {imgs_s.shape}")
    print(
        f"Labels:      {[train_ds_s.classes[label_idx] for label_idx in labels_s.tolist()]}"
    )

    imgs_disp_s = (imgs_s * std + mean).clamp(0, 1)
    grid_s = make_grid(imgs_disp_s, nrow=8, padding=4)
    fig, ax = plt.subplots(figsize=(18, 5))
    _ = ax.imshow(grid_s.permute(1, 2, 0).numpy())
    _ = ax.set_title(
        "ser_sampled — one training batch (224x224, un-normalised)", fontsize=12
    )
    _ = ax.axis("off")
    _ = plt.tight_layout()
    plt.show()